In [0]:
# Configrations
source_dir = "/Volumes/external-catalog/bronze/incremental_load/customers_data/source/"
archive_dir = "/Volumes/external-catalog/bronze/incremental_load/customers_data/archive/"
customers_bronze_table = "`external-catalog`.bronze.customers_bronze"
customers_bronze_error_table = "`external-catalog`.bronze.customers_bronze_error"

print(f"Processing customers data from : {source_dir}")
print(f"Archiving processed data to : {archive_dir}")
print(f"Writing processed data to : {customers_bronze_table}")
print(f"Writing error data to : {customers_bronze_error_table}")

In [0]:
# Import required libraries
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import json
import re

# Define schema for customers data
customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("first_name", StringType(), False),
    StructField("last_name", StringType(), False),
    StructField("email", StringType(), False),
    StructField("phone", StringType(), False),
    StructField("date_of_birth", DateType(), False),
    StructField("registration_date", DateType(), False),
    StructField("address", StringType(), False),
    StructField("city", StringType(), False),
    StructField("state", StringType(), False),
    StructField("zip_code", StringType(), False),
    StructField("country", StringType(), False),
    StructField("customer_tier", StringType(), False),
    StructField("last_login", TimestampType(), False),
    StructField("created_timestamp", TimestampType(), False)
])

print("Schema defined for customers data")


In [0]:
# read and validate customers data

try:
    df_customers = spark.read.schema(customers_schema).csv(source_dir, header=True, dateFormat="yyyy-MM-dd", timestampFormat="yyyy-MM-dd HH:mm:ss")

    df_customers = df_customers.withColumn("processed_timestamp", F.current_timestamp()) \
        .withColumn("batch_id", F.lit(datetime.now().strftime("%Y%m%d_%H%M%S"))) \
        .withColumn("source_system", F.lit("customers"))
    
    total_records = df_customers.count()
    null_customers_ids = df_customers.filter(F.col("customer_id").isNull()).count()
    null_emails = df_customers.filter(F.col("email").isNull()).count()
    null_phonses = df_customers.filter(F.col("phone").isNull()).count()
    future_birth_dates = df_customers.filter(F.col("date_of_birth") > F.current_date()).count()
    null_first_names = df_customers.filter(F.col("first_name").isNull()).count()
    
    invalid_emails = df_customers.filter(F.col("email").isNotNull() & (~F.col("email").contains("@")) | (~F.col("email").contains("."))).count()
    invalid_phones = df_customers.filter(F.col("phone").isNotNull() & (~F.col("phone").contains("-")) | (F.length(F.col("phone")) != 12)).count()
    
    print(f"Total records read: {total_records}")
    print(f"Null customer ids: {null_customers_ids}")
    print(f"Null emails: {null_emails}")
    print(f"Null phones: {null_phonses}")
    print(f"Future birth dates: {future_birth_dates}")
    print(f"Null first names: {null_first_names}")
    print(f"Invalid emails: {invalid_emails}")
    print(f"Invalid phones: {invalid_phones}")

    df_valid_customers = df_customers.filter(F.col("customer_id").isNotNull() & F.col("email").isNotNull() & F.col("phone").isNotNull() & (F.col("date_of_birth") <= F.current_date()) & F.col("first_name").isNotNull())
    df_invalid_customers = df_customers.filter(~F.col("customer_id").isNotNull() | ~F.col("email").isNotNull() | ~F.col("phone").isNotNull() | (F.col("date_of_birth") > F.current_date()) | ~F.col("first_name").isNotNull())

    valid_records = df_valid_customers.count()
    invalid_records = df_invalid_customers.count()

    print(f"Valid records: {valid_records}")
    print(f"Invalid records: {invalid_records}")

except Exception as e:
    print(f"Error reading customers data: {e}")
    raise 

In [0]:
# Data Enrichment - Calculating the customer age and segment

try:
    df_valid_customers = df_valid_customers.withColumn("age", F.floor(F.months_between(F.current_date(), F.col("date_of_birth")) / 12))
    #Create age segment
    df_valid_customers = df_valid_customers.withColumn("customer_segment", F.when(F.col("age") < 25, "Young").when(F.col("age") < 40, "Adult").when(F.col("age") < 60, "Middle-aged").otherwise("Senior"))
    # Calculate days since registration

    df_valid_customers = df_valid_customers.withColumn("days_since_registration", F.datediff(F.current_date(), F.col("registration_date")))
                                                       
    print("Data enrichment completed successfully")

except Exception as e:
    print(f"Error enriching customers data: {e}")
    raise


In [0]:
#Write valid records to bronze delta table
try:
    df_valid_customers.write.mode("overwrite").saveAsTable("customers_bronze")
    print("Valid customers data written to bronze table successfully")

    if invalid_records > 0:
        df_invalid_customers.withColumn("error_reason", F.lit("Data Quality validation failed")) \
            .withColumn("error_timestamp", F.current_timestamp()) \
            .write.format("delta").mode("append").saveAsTable("customers_bronze_error_table")
        print("Invalid customers data written to bronze table successfully")
except Exception as e:
    print(f"Error writing customers data to bronze table: {e}")
    raise


In [0]:
# Archive the processed data
try:
    files = dbutils.fs.ls(source_dir)

    archived_count = 0
    for file in files:
        if file.name.endswith(".csv"):
            src_path = file.path
            archive_path = archive_dir + file.name

            #Move the file to arhive path

            dbutils.fs.mv(src_path, archive_path)
            archived_count += 1

    print(f"Archived {archived_count} files to {archive_dir}")
except Exception as e:
    print(f"Error while archiving processed data : {e}")
    raise

In [0]:
# Log processing summary
processing_summary = {
    "task": "order_bronze_load",
    "timestamp": datetime.now().isoformat(),
    "total_records": total_records,
    "valid_records": valid_records,
    "invalid_records": invalid_records,
    "archived_files": archived_count,
    "status": "SUCCESS" if invalid_records == 0 else "SUCCESS_WITH_WARNINGS"
}

print(f"Processing summary : {processing_summary}")
print(json.dumps(processing_summary, indent=2))

summary_df = spark.createDataFrame([processing_summary])
summary_df.write.mode("append").format("delta").saveAsTable("`external-catalog`.default.processing_control_table")